# 기존 실험 결과 확인
이 노트북은 저장된 집계값을 읽습니다. 재구성한 패키지의 신규 전체 학습 성능은 별도 측정이 필요합니다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository'
sys.path.insert(0, str(ROOT / 'src'))
import torch
torch.set_num_threads(2)
torch.manual_seed(20261112)
print('Repository', ROOT.name, '| PyTorch', torch.__version__)

In [ ]:
import json
from IPython.display import display, Markdown
record = json.loads((ROOT/'results/historical-summary.json').read_text())
print('Recorded at', record['recorded_at'])
print('Test images', record['test_images'], '| precision', record['precision'])

In [ ]:
lines = ['| Model | AP@0.5 | AP@0.75 | Mean image ms |', '|---|---:|---:|---:|']
for model in record['models']:
    lines.append(f"| {model['model']} | {model['ap50']:.4f} | {model['ap75']:.4f} | {model['mean_image_ms']:.1f} |")
display(Markdown('\n'.join(lines)))
print('ViT-S snapshot', record['vits'])

In [ ]:
baseline = record['models'][0]
for model in record['models'][1:]:
    print(model['model'], {'observed_latency_ratio_vs_L':round(baseline['mean_image_ms']/model['mean_image_ms'],3),
        'AP50_delta':round(model['ap50']-baseline['ap50'],5)})
print(*record['limitations'], sep='\n')

AP는 이미지별 Cellpose instance matching 점수의 평균입니다. 지연시간은 이미지별 3회 측정 중앙값의 평균입니다. 표는 해당 평가셋의 관측값이며, ViT-S 최종 평가·반복 측정 분산·GPU 동시 작업 통제 여부를 추가로 확인해야 합니다.